In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import wandb
import matplotlib.pyplot as plt
import numpy as np


In [2]:
class CIFAR10Custom(Dataset):
    def __init__(self, train=True):
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.4914, 0.4822, 0.4465),
                std=(0.247, 0.243, 0.261)
            )
        ])

        self.dataset = torchvision.datasets.CIFAR10(
            root="./data",
            train=train,
            download=True,
            transform=self.transform
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        return image, label


In [3]:
train_loader = DataLoader(
    CIFAR10Custom(train=True),
    batch_size=128,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    CIFAR10Custom(train=False),
    batch_size=128,
    shuffle=False,
    num_workers=2
)


100%|██████████| 170M/170M [00:05<00:00, 29.4MB/s]


In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


In [5]:
# From PyPI
!pip install ptflops

In [6]:
from ptflops import get_model_complexity_info

model = SimpleCNN()
with torch.cuda.device(0):
    flops, params = get_model_complexity_info(
        model,
        (3, 32, 32),
        as_strings=True,
        print_per_layer_stat=False
    )

print("FLOPs:", flops)
print("Params:", params)


FLOPs: 11.25 MMac
Params: 620.81 k


In [7]:
wandb.init(
    project="cifar10-cnn-gradient-flow",
    config={
        "epochs": 30,
        "batch_size": 128,
        "optimizer": "Adam",
        "lr": 1e-3,
        "architecture": "SimpleCNN"
    }
)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pranav-j1414 (pranav-j1414-prom-iit-rajasthan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [9]:
def plot_grad_flow(named_parameters):
    ave_grads = []
    layers = []

    for name, p in named_parameters:
        if p.requires_grad and "bias" not in name:
            layers.append(name)
            ave_grads.append(p.grad.abs().mean().item())

    fig = plt.figure(figsize=(10, 5))
    plt.plot(ave_grads)
    plt.xticks(range(len(layers)), layers, rotation=90)
    plt.xlabel("Layers")
    plt.ylabel("Average Gradient")
    plt.title("Gradient Flow")
    plt.tight_layout()

    return fig


In [10]:
def weight_update_flow(model, prev_weights):
    updates = []

    for (name, p), prev in zip(model.named_parameters(), prev_weights):
        if p.requires_grad:
            updates.append((p - prev).abs().mean().item())

    fig = plt.figure()
    plt.plot(updates)
    plt.title("Weight Update Magnitude")
    plt.xlabel("Layer Index")
    plt.ylabel("Mean |ΔW|")

    return fig


In [ ]:
for epoch in range(30):
    model.train()
    running_loss = 0.0

    prev_weights = [p.clone().detach() for p in model.parameters()]

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()
        running_loss += loss.item()

    grad_fig = plot_grad_flow(model.named_parameters())
    weight_fig = weight_update_flow(model, prev_weights)

    wandb.log({
        "epoch": epoch,
        "train_loss": running_loss / len(train_loader),
        "gradient_flow": wandb.Image(grad_fig),
        "weight_update_flow": wandb.Image(weight_fig)
    })

    plt.close("all")


In [ ]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
wandb.log({"test_accuracy": accuracy})

print(f"Test Accuracy: {accuracy:.2f}%")
